In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import joblib

from sklearn.preprocessing import MinMaxScaler, PowerTransformer, QuantileTransformer
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
file_path = '/content/drive/MyDrive/IoT23Removed/Okiru.csv'

df_real= pd.read_csv(file_path)

df_real.shape

(18840, 23)

In [ ]:
print(df_real['label'].value_counts())

label
Okiru    18840
Name: count, dtype: int64


Optional: Drop Sparse Column

In [ ]:
#Drop Constant/Sparse Columns
drop_cols=['missed_bytes','orig_bytes', 'resp_bytes']

df_real = df_real.drop(columns=drop_cols, errors='ignore')

Log Transformation

In [ ]:
import numpy as np

potential_log_cols = ['duration', 'orig_bytes', 'resp_bytes', 'orig_ip_bytes', 'iat', 'conn_rate_1s']
columns_to_log = [c for c in potential_log_cols if c in df_real.columns]

# Apply the log(1 + X) transformation
for col in columns_to_log:
    df_real[col + '_log'] = np.log1p(df_real[col])

#drop the original columns and use the new '_log' columns
df_real = df_real.drop(columns=columns_to_log)

Box Cox Transformation

In [ ]:
boxcox_cols = ['orig_bytes', 'resp_bytes','orig_pkts', 'resp_pkts'] #'duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts'

# 2. Dictionary to store the transformer for each column
transformers = {}

for col in boxcox_cols:
    if col in df_real.columns:
        print(f"Transforming {col}...")

        # A. Handle Zeros: Box-Cox requires strictly positive values (>0)
        #  add 1 (just like you did for duration)
        df_real[col] = df_real[col] + 1

        # B. Initialize the transformer
        pt = PowerTransformer(method='box-cox')

        # C. Fit and Transform
        # Note: We name the new column with a suffix so we can track it
        df_real[f'{col}_boxcox'] = pt.fit_transform(df_real[[col]])

        # D. Save the transformer object to our dictionary
        transformers[col] = pt

        # E. Drop the original column
        df_real = df_real.drop(columns=[col])

# 3. Save the ENTIRE dictionary of transformers to one file
# You will need this file to reverse the process later!
with open('boxcox_transformers.pkl', 'wb') as f:
    pickle.dump(transformers, f)

Transforming orig_pkts...


Quantile Transformation

In [ ]:
df_real.columns

Index(['duration', 'orig_pkts', 'orig_ip_bytes', 'proto_icmp', 'proto_tcp',
       'proto_udp', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO',
       'conn_state_RSTOS0', 'conn_state_RSTR', 'conn_state_RSTRH',
       'conn_state_S0', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH',
       'conn_rate_1s', 'unique_ports_1m', 'iat', 'label'],
      dtype='object')

In [ ]:
quantile_cols = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'orig_ip_bytes','conn_rate_1s', 'unique_ports_1m', 'iat','missed_bytes']
                  #'duration', 'orig_bytes', 'resp_bytes', 'orig_pkts','orig_ip_bytes','conn_rate_1s','unique_ports_1m', 'iat', missed_bytes
transformer = {}

for col in quantile_cols:
    if col in df_real.columns:
        print(f"Transforming {col}...")

        qt = QuantileTransformer(output_distribution='normal',
                                 n_quantiles=2000,
                                 random_state=42)

        df_real[f'{col}_quantile'] = qt.fit_transform(df_real[[col]])

        transformer[col] = qt

        df_real = df_real.drop(columns=[col])

# Save the transformer
with open('quantile_transformer.pkl', 'wb') as f:
    pickle.dump(transformer, f)


Transforming duration...
Transforming orig_pkts...
Transforming orig_ip_bytes...
Transforming conn_rate_1s...
Transforming unique_ports_1m...
Transforming iat...


Separate Column

In [ ]:
#Remove label column
label_value = df_real['label']
df_features = df_real.drop('label', axis=1)

Scaling (MinMax)

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
data = scaler.fit_transform(df_features)

Build WGAN-GP Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# HYPERPARAMETER
GEN_LEARNING_RATE = 5e-5
CRIT_LEARNING_RATE = 1e-4
BATCH_SIZE = 64
Z_DIM = 32
NUM_EPOCHS = 300
FEATURES_CRITIC = 64
FEATURES_GEN = 64
CRITIC_ITERATIONS = 5
LAMBDA_GP = 10

Using device: cpu


In [ ]:
class Generator(nn.Module):
    def __init__(self, z_dim, features_gen, output_dim):
        super(Generator, self).__init__()
        self.gen = nn.Sequential(
            # Layer 1
            nn.Linear(z_dim, features_gen * 4),
            nn.LayerNorm(features_gen * 4),
            nn.LeakyReLU(0.2),

            #Layer 2
            nn.Linear(features_gen * 4, features_gen * 2),
            nn.LayerNorm(features_gen * 2),
            nn.LeakyReLU(0.2),

            #Layer 3
            nn.Linear(features_gen * 2, features_gen),
            nn.LayerNorm(features_gen),
            nn.LeakyReLU(0.2),

            #Output Layer
            nn.Linear(features_gen, output_dim),
            nn.Tanh(), # force output to be between -1 and 1
        )

    def forward(self, x):
        return self.gen(x)

# Get the number of features from the preprocessed data
input_dim = data.shape[1]

print("Generator class defined.")

Generator class defined.


In [ ]:
class Critic(nn.Module):
    def __init__(self, input_dim, features_critic):
        super(Critic, self).__init__()
        self.critic = nn.Sequential(
            # Input: data (real or fake) with input_dim features
            # Layer 1
            nn.Linear(input_dim, features_critic*4),
            nn.LeakyReLU(0.2),

            #Layer 2
            nn.Linear(features_critic * 4, features_critic * 2),
            nn.LeakyReLU(0.2),

            #Layer 3
            nn.Linear(features_critic * 2, features_critic),
            nn.LeakyReLU(0.2),

            #Output Layer
            nn.Linear(features_critic, 1),
            # No activation function for critic
        )

    def forward(self, x):
        return self.critic(x)

print("Critic class defined.")

Critic class defined.


In [ ]:
dataset = TensorDataset(torch.tensor(data, dtype=torch.float32))
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize Models and Optimizers
gen = Generator(Z_DIM, FEATURES_GEN, input_dim).to(device)
critic = Critic(input_dim, FEATURES_CRITIC).to(device)

# Adam Optimizers
opt_gen = optim.Adam(gen.parameters(), lr=GEN_LEARNING_RATE, betas=(0.0, 0.9))
opt_critic = optim.Adam(critic.parameters(), lr=CRIT_LEARNING_RATE, betas=(0.0, 0.9))

In [ ]:
data.max()

np.float64(1.0000000000000002)

In [ ]:
#Gradient Penalty
def calculate_gradient_penalty(critic, real_data, fake_data, device):
    batch_size, features = real_data.shape

    # Generate random epsilon for interpolation
    epsilon = torch.rand(batch_size, 1).to(device)

    # Create interpolated images (between real and fake)
    # Expand epsilon to match data shape
    epsilon = epsilon.expand_as(real_data)
    interpolated = epsilon * real_data + (1 - epsilon) * fake_data

    # Require gradients for interpolated data
    interpolated = interpolated.requires_grad_(True)

    # Calculate critic scores for interpolated data
    critic_interpolated = critic(interpolated)

    # Calculate gradients of the scores with respect to the interpolated data
    gradients = autograd.grad(
        outputs=critic_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interpolated),
        create_graph=True,
        retain_graph=True,
    )[0]

    # Flatten gradients to calculate norm
    gradients = gradients.view(batch_size, -1)

    # Calculate gradient norm
    gradient_norm = gradients.norm(2, dim=1)

    # Calculate penalty: (norm - 1)^2
    gradient_penalty = ((gradient_norm - 1) ** 2).mean()

    return gradient_penalty

Training Loop

In [ ]:
gen.train()
critic.train()

# Lists to keep track of progress
gen_losses = []
critic_losses = []
gp_losses = []

#cycle through data for the k Critic steps
data_iter = iter(loader)

for epoch in range(NUM_EPOCHS):
    epoch_gp = 0
    batch_count = 0

    # Iterate through the data one batch at a time
    for batch_idx, real in enumerate(loader):
        batch_count += 1

        # ==============================
        #  Train Critic: CRITIC_ITERATIONS (k) times
        # ==============================
        for _ in range(CRITIC_ITERATIONS):

            # --- Get Fresh Data (Crucial for the k steps) ---
            try:
                # Try to get the next fresh batch
                real, = next(data_iter)
            except StopIteration:
                # If we run out of data, restart the iterator
                data_iter = iter(loader)
                real, = next(data_iter) # Grab the first batch of the "new epoch"

            real = real.to(device)
            cur_batch_size = real.shape[0]

            # Generate fake data and detach from gen's graph
            noise = torch.randn(cur_batch_size, Z_DIM).to(device)
            fake = gen(noise).detach()

            # Critic forward pass
            critic_real = critic(real)
            critic_fake = critic(fake).reshape(-1)

            # Calculate Gradient Penalty (GP)
            gp = calculate_gradient_penalty(critic, real, fake, device)
            epoch_gp += gp.item() #capture gp value

            # Critic Loss (Wasserstein distance + GP)
            loss_critic = (
                -(torch.mean(critic_real) - torch.mean(critic_fake)) + LAMBDA_GP * gp
            )

            # Critic Backward pass and optimization step
            critic.zero_grad()
            loss_critic.backward()
            opt_critic.step()

        # ==============================
        #  Train Generator: 1 time
        # ==============================
        # NOTE: This runs once for every CRITIC_ITERATIONS set of Critic updates

        noise = torch.randn(BATCH_SIZE, Z_DIM).to(device)
        fake = gen(noise)
        critic_fake = critic(fake).reshape(-1)

        # Generator Loss: -E[D(fake)] (Maximize E[D(fake)] by minimizing -E[D(fake)])
        loss_gen = -torch.mean(critic_fake)

        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

    # --- Logging ---
    # 1. Safety check for the denominator
    denominator = batch_count * CRITIC_ITERATIONS

    if denominator > 0:
        avg_gp = epoch_gp / denominator
    else:
        avg_gp = 0.0  # Default to 0 if no batches were processed
        print(f"Warning: No batches processed in Epoch {epoch}")

    # Append losses to lists (ONLY ONCE PER EPOCH)
    gen_losses.append(loss_gen.item())
    critic_losses.append(loss_critic.item())
    gp_losses.append(avg_gp)

    if epoch % 5 == 0:
        print(
            f"Epoch [{epoch}/{NUM_EPOCHS}] \t "
            f"C Loss: {loss_critic.item():.4f} \t "
            f"G Loss: {loss_gen.item():.4f} \t"
            f"GP : {avg_gp:.4f}"
        )

        #CHECKPOINT
        checkpoint={
            'epoch': epoch,
            'gen_state_dict': gen.state_dict(),
            'critic_state_dict': critic.state_dict(),
            'opt_gen_state_dict': opt_gen.state_dict(),
            'opt_critic_state_dict': opt_critic.state_dict(),
        }
        torch.save(checkpoint, f"epoch_{epoch}.pth")


Epoch [0/300] 	 C Loss: -0.7158 	 G Loss: -2.1824 	GP : 0.0350
Epoch [5/300] 	 C Loss: -0.0259 	 G Loss: 0.1196 	GP : 0.0019
Epoch [10/300] 	 C Loss: -0.0906 	 G Loss: 1.5518 	GP : 0.0015
Epoch [15/300] 	 C Loss: -0.0422 	 G Loss: 0.7169 	GP : 0.0013
Epoch [20/300] 	 C Loss: -0.0622 	 G Loss: 0.1314 	GP : 0.0011
Epoch [25/300] 	 C Loss: -0.0681 	 G Loss: 0.1030 	GP : 0.0009
Epoch [30/300] 	 C Loss: -0.0651 	 G Loss: -0.1400 	GP : 0.0008
Epoch [35/300] 	 C Loss: -0.0424 	 G Loss: -0.3169 	GP : 0.0007
Epoch [40/300] 	 C Loss: 0.0059 	 G Loss: -0.5699 	GP : 0.0007
Epoch [45/300] 	 C Loss: -0.0875 	 G Loss: -0.2735 	GP : 0.0007
Epoch [50/300] 	 C Loss: -0.0979 	 G Loss: -0.6839 	GP : 0.0006
Epoch [55/300] 	 C Loss: -0.0850 	 G Loss: -0.6871 	GP : 0.0006
Epoch [60/300] 	 C Loss: -0.0532 	 G Loss: -0.7574 	GP : 0.0006
Epoch [65/300] 	 C Loss: -0.0684 	 G Loss: -0.6812 	GP : 0.0006
Epoch [70/300] 	 C Loss: -0.0011 	 G Loss: -0.8257 	GP : 0.0006
Epoch [75/300] 	 C Loss: -0.0127 	 G Loss: -0.70

KeyboardInterrupt: 

In [ ]:
import csv

# --- SAVE TO CSV FOR STREAMLIT ---
with open('Okiru_training_logs.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'd_loss', 'g_loss', 'gp'])

    for i in range(len(gen_losses)):
        # Use f-strings to format each float to 4 decimal places
        writer.writerow([
            i,
            f"{critic_losses[i]:.4f}",
            f"{gen_losses[i]:.4f}",
            f"{gp_losses[i]:.4f}"
        ])

print("Training logs saved to training_logs.csv")

Training logs saved to training_logs.csv


Generate and Post Process Synthetic Data

In [ ]:
ORIGINAL_DATA = 18840
TARGET_DATA = 80000
NUM_SAMPLES_TO_GENERATE = TARGET_DATA - ORIGINAL_DATA

best_gen = Generator(Z_DIM, FEATURES_GEN, input_dim).to(device)


checkpoint_path = "epoch_135.pth" #--change the name
checkpoint = torch.load(checkpoint_path)

# Load the weights into the model
best_gen.load_state_dict(checkpoint['gen_state_dict'])
best_gen.eval()

print(f"Successfully loaded model from {checkpoint_path}")

print(f"Generating {NUM_SAMPLES_TO_GENERATE} synthetic samples...")

with torch.no_grad():
    # 3. Create random noise (Latent Vector)
    noise = torch.randn(NUM_SAMPLES_TO_GENERATE, Z_DIM).to(device)

    # 4. Generate Fake Data (Values are still between -1 and 1)
    fake_data_scaled = best_gen(noise)

    # 5. Move to CPU to work with Scikit-Learn/Pandas
    fake_data_scaled = fake_data_scaled.cpu().numpy()

# 6. Inverse Transfor (MinMax)----------------------
fake_data_unscaled = scaler.inverse_transform(fake_data_scaled)

# 7. Create DataFrame ------------------------------
df_synthetic = pd.DataFrame(fake_data_unscaled, columns=df_features.columns)

print("Cleaning synthetic data...")

# #8.---------------------------------------------Reverse Log Transformation ----------------------------------
# log_transformed_cols = [c for c in df_synthetic.columns if '_log' in c]

# for col in log_transformed_cols:
#     original_col_name = col.replace('_log', '')

#     # Apply the inverse transformation: np.expm1(X) = e^X - 1
#     df_synthetic[original_col_name] = np.expm1(df_synthetic[col])

# # Drop the log-transformed columns (they are now redundant)
# df_synthetic = df_synthetic.drop(columns=log_transformed_cols, errors='ignore')


# # 9. ----------------------------------------------Reverse Box Cox for Transformation ------------------------------------------------
# # Load dictionary
# with open('boxcox_transformers.pkl', 'rb') as f:
#     transformers = pickle.load(f)

# # Find all columns that were Box-Cox
# boxcox_cols_synthetic = [c for c in df_synthetic.columns if c.endswith('_boxcox')]

# for col in boxcox_cols_synthetic:
#     original_name = col.replace('_boxcox', '')

#     if original_name in transformers:
#         # Get the specific transformer
#         pt = transformers[original_name]

#         val_reshaped = df_synthetic[[col]].values

#         # Inverse Transform
#         df_synthetic[original_name] = pt.inverse_transform(val_reshaped)

#         # Subtract 1 (to undo the +1 we added in preprocessing)
#         df_synthetic[original_name] = df_synthetic[original_name] - 1

#         # Drop the temporary '_boxcox' column
#         df_synthetic = df_synthetic.drop(columns=[col])


# 10. ----------------------------------------------Reverse Quantile Transformation ------------------------------------------------
# Load saved transformer
with open('quantile_transformer.pkl', 'rb') as f:
    transformers = pickle.load(f)

quantile_cols_synthetic = [c for c in df_synthetic.columns if c.endswith('_quantile')]

for col in quantile_cols_synthetic:
    original_name = col.replace('_quantile', '')

    if original_name in transformers:
        print(f"Inversing {col}...")

        # A. Get the specific transformer for this column
        qt = transformers[original_name]

        # B. Reshape for sklearn (needs 2D array)
        val_reshaped = df_synthetic[[col]].values

        # C. Inverse Transform
        # Note: No need to subtract 1 because QuantileTransformer handles zeros naturally
        df_synthetic[original_name] = qt.inverse_transform(val_reshaped)

        # D. Drop the temporary '_quantile' column
        df_synthetic = df_synthetic.drop(columns=[col])
# -------------------------------------------------------------------------------------------

#11. Ensure non-negative values
non_negative_cols = ['duration', 'orig_bytes', 'resp_bytes', 'orig_pkts', 'resp_pkts']
for col in non_negative_cols:
    if col in df_synthetic.columns:
        df_synthetic[col] = df_synthetic[col].clip(lower=0)


#12. Fix OHE
ohe_groups = ['proto', 'conn_state']

for prefix in ohe_groups:
    cols = [c for c in df_synthetic.columns if c.startswith(prefix)]

    if cols:
        # 1. Find the column with the maximum predicted value (closest to 1) for each row
        max_cols = df_synthetic[cols].idxmax(axis=1)

        # 2. Reset all OHE columns for this group to 0
        df_synthetic[cols] = 0

        # 3. Set the column identified by idxmax to 1
        for col_name in cols:
            df_synthetic.loc[max_cols == col_name, col_name] = 1


#13. Final Integer / Rounding Fixes
integer_columns = ['orig_pkts', 'resp_pkts','orig_bytes', 'resp_bytes']#'orig_pkts', 'resp_pkts','orig_bytes', 'resp_bytes'
categorical_cols = [c for c in df_synthetic.columns if any(c.startswith(p) for p in ohe_groups)]
integer_columns.extend(categorical_cols)

# Apply rounding and final type conversion
for col in integer_columns:
    if col in df_synthetic.columns:
        df_synthetic[col] = df_synthetic[col].round().astype(np.int64) # Use np.int64 for safety


# 14. Re-attach the Label
df_synthetic['label'] = 'Okiru'   #--NEED TO CHANGE THIS

# 15. Restore dropped columns - Optional
df_synthetic['missed_bytes']=0.0  #--NEED TO CONSIDER BASED ON DATASET
df_synthetic['orig_bytes']=0.0 #okiru
df_synthetic['resp_bytes']=0.0 #okiru

#Save the Cleaned Synthetic Data
df_synthetic.to_csv('/content/drive/MyDrive/IoT23Removed/Okiru_synthetic375_80k.csv', index=False)

print("Synthetic data generation and cleaning complete!")
df_synthetic.head(10)

Successfully loaded model from epoch_135.pth
Generating 61160 synthetic samples...
Cleaning synthetic data...
Inversing duration_quantile...
Inversing orig_pkts_quantile...
Inversing orig_ip_bytes_quantile...
Inversing conn_rate_1s_quantile...
Inversing unique_ports_1m_quantile...
Inversing iat_quantile...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but QuantileTransformer was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12

Synthetic data generation and cleaning complete!


,proto_icmp,proto_tcp,proto_udp,conn_state_OTH,conn_state_REJ,conn_state_RSTO,conn_state_RSTOS0,conn_state_RSTR,conn_state_RSTRH,conn_state_S0,...,duration,orig_pkts,orig_ip_bytes,conn_rate_1s,unique_ports_1m,iat,label,missed_bytes,orig_bytes,resp_bytes
0,0,1,0,0,0,0,0,0,0,1,...,0.000002,2,80.0,283.776978,7.0,0.000004,Okiru,0.0,0.0,0.0
1,0,1,0,0,0,0,0,0,0,1,...,0.000005,2,80.0,583.000000,7.0,0.574939,Okiru,0.0,0.0,0.0
2,0,1,0,0,0,0,0,0,0,1,...,0.000002,2,80.0,565.000000,7.0,0.000004,Okiru,0.0,0.0,0.0
3,0,1,0,0,0,0,0,0,0,1,...,0.000006,2,80.0,565.000000,7.0,0.000220,Okiru,0.0,0.0,0.0
4,0,1,0,0,0,0,0,0,0,1,...,0.000002,2,80.0,592.000000,7.0,0.000004,Okiru,0.0,0.0,0.0
5,0,1,0,0,0,0,0,0,0,1,...,0.000005,2,80.0,590.000000,7.0,0.014036,Okiru,0.0,0.0,0.0
6,0,1,0,0,0,0,0,0,0,1,...,0.000006,2,80.0,627.000000,7.0,0.000234,Okiru,0.0,0.0,0.0
7,0,1,0,0,0,0,0,0,0,1,...,0.000002,2,80.0,497.000000,7.0,0.000004,Okiru,0.0,0.0,0.0
8,0,1,0,0,0,0,0,0,0,1,...,0.000002,2,80.0,563.000000,7.0,0.000004,Okiru,0.0,0.0,0.0
9,0,1,0,0,0,0,0,0,0,1,...,0.000002,2,80.0,630.000000,7.0,0.000007,Okiru,0.0,0.0,0.0


In [ ]:
df_synthetic.shape

(247064, 21)

Save Model

In [ ]:
import torch
import joblib  # Standard library for saving Scikit-Learn object

# --- 1. Save the PyTorch Models ---
torch.save(gen.state_dict(), 'generator.pth')
torch.save(critic.state_dict(), 'critic.pth')

# --- 2. Save the Scaler ---
# CRITICAL: If you lose this, your generator's output will be meaningless numbers
joblib.dump(scaler, 'scaler.gz')

print("Models and Scaler saved successfully!")
print("Files created: 'generator.pth', 'critic.pth', 'scaler.gz'")